In [1]:
import scipy.io
adjacency_matrices = scipy.io.loadmat('/Users/siddharth/Library/CloudStorage/OneDrive-IndianInstituteofTechnology(BHU),Varanasi/Mac_Data/Pendrive data/Braindata.mat')['graph']

In [2]:
import numpy as np
adjacency_matrices = np.reshape(adjacency_matrices, (2300, 306, 306))
labels = scipy.io.loadmat('/Users/siddharth/Library/CloudStorage/OneDrive-IndianInstituteofTechnology(BHU),Varanasi/Mac_Data/Pendrive data/Braindata.mat')['label']
print(adjacency_matrices.shape)
print(labels.shape)

(2300, 306, 306)
(1, 2300)


In [1]:
#This code is when you dont have data and you want to test model
import numpy as np

# Generate random adjacency matrices with shape (2300, 306, 306)
adjacency_matrices = np.random.randint(0, 2300, (2300, 306, 306))

# Generate random labels with shape (1, 2300)
labels = np.random.randint(0, 2, (1, 2300))

# Print the shapes to confirm
print("Adjacency Matrices Shape:", adjacency_matrices.shape)
print("Labels Shape:", labels.shape)


Adjacency Matrices Shape: (2300, 306, 306)
Labels Shape: (1, 2300)


In [2]:
labels=np.transpose(labels,(1,0))


In [7]:
# import numpy as np
# adjacency_matrices=np.transpose(adjacency_matrices, (2,0,1))
labels.shape

(2300, 1)

In [3]:
adjacency_matrices.shape

(2300, 306, 306)

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, Dataset, DataLoader
from torch_geometric.nn import GCNConv, GATConv, ChebConv, SAGEConv, global_mean_pool
from scipy.io import loadmat
from sklearn.model_selection import train_test_split

/Users/siddharth/opt/anaconda3/lib/python3.9/site-packages/torch_geometric/typing.py:31: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: dlopen(/Users/siddharth/opt/anaconda3/lib/python3.9/site-packages/torch_scatter/_scatter_cpu.so, 0x0006): Symbol not found: __ZN2at4_ops6narrow4callERKNS_6TensorExxx
  Referenced from: <FC4F5CE6-3038-3A9A-B98B-661CD724F01B> /Users/siddharth/opt/anaconda3/lib/python3.9/site-packages/torch_scatter/_scatter_cpu.so
  Expected in:     <66FB8649-BB87-3CD6-A177-462038DCAE02> /Users/siddharth/opt/anaconda3/lib/python3.9/site-packages/torch/lib/libtorch_cpu.dylib
  warnings.warn(f"An issue occurred while importing 'torch-scatter'. "


In [5]:
class BrainConnectivityDataset(Dataset):
    def __init__(self, adjacency_matrices, labels, transform=None, pre_transform=None):
        self.adjacency_matrices = adjacency_matrices
        self.labels = labels
        super(BrainConnectivityDataset, self).__init__(None, transform, pre_transform)

    def len(self):
        return len(self.labels)

    def get(self, idx):
        edge_index = self.adjacency_matrix_to_edge_index(self.adjacency_matrices[idx])
        x = torch.ones((self.adjacency_matrices.shape[1], 1), dtype=torch.float)
        y = torch.tensor([self.labels[idx]], dtype=torch.long)
        data = Data(x=x, edge_index=edge_index, y=y)
        return data

    @staticmethod
    def adjacency_matrix_to_edge_index(matrix):
        edge_index = []
        for i in range(matrix.shape[0]):
            for j in range(matrix.shape[1]):
                if matrix[i, j]:  # If there is an edge
                    edge_index.append([i, j])
        return torch.tensor(edge_index, dtype=torch.long).t().contiguous()

# # Load the .mat files
# labels = loadmat('path_to_labels.mat')['variable_name'].squeeze()

# Split the dataset into training and testing sets (Thos os no)
train_indices, test_indices = train_test_split(range(len(labels)), test_size=0.2, random_state=42)
train_data = adjacency_matrices[train_indices]
test_data = adjacency_matrices[test_indices]
train_labels = labels[train_indices]
test_labels = labels[test_indices]

# Create the datasets
train_dataset = BrainConnectivityDataset(train_data, train_labels)
test_dataset = BrainConnectivityDataset(test_data, test_labels)

In [6]:
print(test_dataset)
print(train_dataset)

BrainConnectivityDataset(460)
BrainConnectivityDataset(1840)


In [8]:
class BrainConnectivityNet(nn.Module):
    def __init__(self, num_nodes, num_classes):
        super(BrainConnectivityNet, self).__init__()
        # GCN branch
        self.gcn1 = GCNConv(num_nodes, 126)
        self.gcn2 = GCNConv(126, 96)

        # GAT branch
        self.gat1 = GATConv(num_nodes, 126)
        self.gat2 = GATConv(126, 96)

        # Chebyshev branch
        self.cheb1 = ChebConv(num_nodes, 126, K=2)
        self.cheb2 = ChebConv(126, 96, K=2)

        # GraphSAGE branch
        self.sage1 = SAGEConv(num_nodes, 126)
        self.sage2 = SAGEConv(126, 96)

        # Pooling layer
        self.pool = global_mean_pool
        # intermediate_layer_model = model(inputs=model.input,
        #                                   outputs=model.get_layer(gcn1).output)
        # Classifier
        self.classifier = nn.Linear(96*4,num_classes)  # 4 branches with 64 features each

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        # Node features are all ones
        x = torch.ones((data.num_nodes, 306), dtype=torch.float)

        # GCN branch
        x_gcn = F.relu(self.gcn1(x, edge_index))
        x_gcn = F.dropout(x_gcn, training=self.training)
        x_gcn = F.relu(self.gcn2(x_gcn, edge_index))

        # GAT branch
        x_gat = F.relu(self.gat1(x, edge_index))
        x_gat = F.dropout(x_gat, training=self.training)
        x_gat = F.relu(self.gat2(x_gat, edge_index))

        # Chebyshev branch
        x_cheb = F.relu(self.cheb1(x, edge_index))
        x_cheb = F.dropout(x_cheb, training=self.training)
        x_cheb = F.relu(self.cheb2(x_cheb, edge_index))

        # GraphSAGE branch
        x_sage = F.relu(self.sage1(x, edge_index))
        x_sage = F.dropout(x_sage, training=self.training)
        x_sage = F.relu(self.sage2(x_sage, edge_index))

        # Concatenate the outputs from all branches
        x = torch.cat((x_gcn, x_gat, x_cheb, x_sage), dim=1)

        # Apply the pooling layer to get graph-level representation
        x = self.pool(x, batch)

        # Apply the classification layer
        x = self.classifier(x)

        return F.log_softmax(x, dim=1)



In [9]:
# Create DataLoaders for training and testing
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

/Users/siddharth/opt/anaconda3/lib/python3.9/site-packages/torch_geometric/deprecation.py:22: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


In [10]:
# Initialize the model
model = BrainConnectivityNet(num_nodes=306, num_classes=4)

# Define the optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [11]:
import cv2
import numpy as np
# Training loop
i =1
model.train()
for epoch in range(200):
    total_loss = 0
    for data in train_loader:
        optimizer.zero_grad()
        output = model(data)
        loss = F.nll_loss(output, data.y.view(-1))
        loss.backward()
        optimizer.step()
        
        # img = cv2.imread('resize.png',0)
        # img = np.reshape(img, (1,800,64,1)) # (n_images, x_shape, y_shape, n_channels)
        # img.shape 
        # intermediate_output = intermediate_layer_model.predict(img)
        # print(intermediate_output)
        total_loss += loss.item()
        # print(f'Data {i} Running')
        i = i+1
    print(f'Epoch {epoch+1}, Loss: {total_loss / len(train_loader)}')

/var/folders/dh/nfwywq5x6vj6wb3fkdkwkg000000gn/T/ipykernel_98236/2970887910.py:13: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:264.)
  y = torch.tensor([self.labels[idx]], dtype=torch.long)


KeyboardInterrupt: 

In [ ]:
# i=0
for data in train_loader:
    print(data.y)
    # print(data.x)

In [20]:
class GNN(torch.nn.Module):
    def __init__(self, feature_size, model_params):
        super(GNN, self).__init__()
        embedding_size = model_params["model_embedding_size"]
        n_heads = model_params["model_attention_heads"]
        self.n_layers = model_params["model_layers"]
        dropout_rate = model_params["model_dropout_rate"]
        top_k_ratio = model_params["model_top_k_ratio"]
        self.top_k_every_n = model_params["model_top_k_every_n"]
        dense_neurons = model_params["model_dense_neurons"]
        edge_dim = model_params["model_edge_dim"]

        self.conv_layers = ModuleList([])
        self.transf_layers = ModuleList([])
        self.pooling_layers = ModuleList([])
        self.bn_layers = ModuleList([])

        # Transformation layer
        self.conv1 = TransformerConv(feature_size, 
                                    embedding_size, 
                                    heads=n_heads, 
                                    dropout=dropout_rate,
                                    edge_dim=edge_dim,
                                    beta=True) 

        self.transf1 = Linear(embedding_size*n_heads, embedding_size)
        self.bn1 = BatchNorm1d(embedding_size)

        # Other layers
        for i in range(self.n_layers):
            self.conv_layers.append(TransformerConv(embedding_size, 
                                                    embedding_size, 
                                                    heads=n_heads, 
                                                    dropout=dropout_rate,
                                                    edge_dim=edge_dim,
                                                    beta=True))

            self.transf_layers.append(Linear(embedding_size*n_heads, embedding_size))
            self.bn_layers.append(BatchNorm1d(embedding_size))
            if i % self.top_k_every_n == 0:
                self.pooling_layers.append(TopKPooling(embedding_size, ratio=top_k_ratio))
            

        # Linear layers
        self.linear1 = Linear(embedding_size*2, dense_neurons)
        self.linear2 = Linear(dense_neurons, int(dense_neurons/2))  
        self.linear3 = Linear(int(dense_neurons/2), 1)  

    def forward(self, x, edge_attr, edge_index, batch_index):
        # Initial transformation
        x = self.conv1(x, edge_index, edge_attr)
        x = torch.relu(self.transf1(x))
        x = self.bn1(x)

        # Holds the intermediate graph representations
        global_representation = []

        for i in range(self.n_layers):
            x = self.conv_layers[i](x, edge_index, edge_attr)
            x = torch.relu(self.transf_layers[i](x))
            x = self.bn_layers[i](x)
            # Always aggregate last layer
            if i % self.top_k_every_n == 0 or i == self.n_layers:
                x , edge_index, edge_attr, batch_index, _, _ = self.pooling_layers[int(i/self.top_k_every_n)](
                    x, edge_index, edge_attr, batch_index
                    )
                # Add current representation
                global_representation.append(torch.cat([gmp(x, batch_index), gap(x, batch_index)], dim=1))
    
        x = sum(global_representation)

        # Output block
        x = torch.relu(self.linear1(x))
        x = F.dropout(x, p=0.8, training=self.training)
        x = torch.relu(self.linear2(x))
        x = F.dropout(x, p=0.8, training=self.training)
        x = self.linear3(x)

        return x